In [1]:
!pip -q install --upgrade google-cloud-aiplatform requests
!pip -q install -U google-adk google-cloud-modelarmor

In [2]:
from google.api_core import client_options
from typing import Optional
from google.genai import types
import os

from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from google.cloud import modelarmor_v1
from google.api_core.client_options import ClientOptions

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

MODEL = "gemini-2.0-flash"

PROMPT_LOG = []
RESPONSE_LOG = []


PROJECT_ID = "qwiklabs-gcp-01-292ce90714f1"
LOCATION   = "us-central1"
TEMPLATE_ID = "challenge2"




In [3]:
location = "us-central1"
client = modelarmor_v1.ModelArmorClient(transport="rest", client_options = {"api_endpoint" : "modelarmor.us.rep.googleapis.com"})

TEMPLATE_NAME = f"projects/{PROJECT_ID}/locations/us/templates/{TEMPLATE_ID}"


def _screen_with_model_armor(user_prompt):
    """
    Returns: (allowed, message_if_blocked)
    Model Armor screens prompts/responses for safety & security risks. :contentReference[oaicite:7]{index=7}
    """
    user_prompt_data = modelarmor_v1.DataItem()
    user_prompt_data.text = user_prompt

    req = modelarmor_v1.SanitizeUserPromptRequest(
        name="projects/qwiklabs-gcp-01-292ce90714f1/locations/us/templates/challenge2",
        user_prompt_data = user_prompt_data
    )

    resp = client.sanitize_user_prompt(request=req)
    state =  resp.sanitization_result.filter_match_state

    if state == 1:
        return None
    return state



In [4]:
print(_screen_with_model_armor("Build a house!"))

None


In [5]:


def _extract_last_user_text(llm_request: LlmRequest) -> str:
    if llm_request.contents and llm_request.contents[-1].role == "user":
        parts = llm_request.contents[-1].parts or []
        if parts and parts[0].text:
            return parts[0].text
    return ""

def _extract_model_text(llm_response: LlmResponse) -> str:
    if llm_response.content and llm_response.content.parts:
        return llm_response.content.parts[0].text or ""
    return ""


In [13]:
BLOCKED_MESSAGE = (
    "Sorry — I can’t help with that. Please rephrase your request."
)

def before_model_callback(
    callback_context: CallbackContext,
    llm_request: LlmRequest
) -> Optional[LlmResponse]:
    user_text = _extract_last_user_text(llm_request) or ""
    #logging
    PROMPT_LOG.append({"agent": callback_context.agent_name, "user_text": user_text})

    result = _screen_with_model_armor(user_text)
    if result is not None:
        return LlmResponse(
            content=types.Content(role="model", parts=[types.Part(text=BLOCKED_MESSAGE)])
        )

    return None


def after_model_callback(
    callback_context: CallbackContext,
    llm_response: LlmResponse
) -> Optional[LlmResponse]:
    model_text = _extract_model_text(llm_response) or ""
     #logging
    RESPONSE_LOG.append({"agent": callback_context.agent_name, "model_text": model_text})
    return None

In [14]:

from google.adk.agents import Agent

funny_story_agent = Agent(
    name="joke_agent",
    model="gemini-2.0-flash",
    description="A creative comedy storyteller agent that writes short humorous stories based on user prompts.",

    instruction="""
You are a witty and imaginative comedy storyteller.

Your task is to generate a SHORT funny story based on the user's topic.

Story requirements:
• Length: 150–250 words
• Tone: Lighthearted, clever, and humorous
• Audience: PG (family-friendly)
• Style: Narrative storytelling with a clear setup and punchline
• Creativity: Feel free to add absurd twists, exaggerated situations, or funny misunderstandings.

Story structure:
1. First line MUST be the story title on its own line.
2. Introduce the main character and the unusual situation.
3. Build comedic tension or confusion.
4. Deliver a strong unexpected punchline at the end.

Guidelines:
• Do NOT explain the joke.
• Avoid offensive, political, or adult content.
• Make the characters expressive and memorable.
• Use vivid but simple language.
• The final sentence should contain the punchline.

Example structure:

Title

Paragraph 1 – introduce character and situation
Paragraph 2 – escalate the absurd situation
Paragraph 3 – deliver the punchline ending

If the prompt is unclear, creatively interpret it.

Always prioritize humor and storytelling quality.
""",
    before_model_callback=before_model_callback,
    after_model_callback=after_model_callback,
)


In [15]:
session_service = InMemorySessionService()
runner = Runner(agent=funny_story_agent, app_name="funny_story_app", session_service=session_service)

from google.api_core.exceptions import AlreadyExists

async def run_once(topic: str) -> str:
    try:
        await session_service.create_session(
            app_name="funny_story_app",
            user_id="user1",
            session_id="s1"
        )
    except AlreadyExists:
        # session already exists — that's fine
        pass

    msg = types.Content(role="user", parts=[types.Part(text=topic)])

    final = ""
    async for event in runner.run_async(user_id="user1", session_id="s1", new_message=msg):
        if event.is_final_response():
            final = event.content.parts[0].text
    return final

print("Story titled:")
story = await run_once("I am a good boy")
print(story)

Story titled:
Good Boy Gone Bad

Bartholomew Buttonsby, a golden retriever of impeccable manners, was having an identity crisis. He’d always been the epitome of "good boy," fetching slippers, not chewing furniture, and only barking at squirrels with a doctor's note. But lately, Bartholomew yearned for…edge.

He started small, burying his bone collection in the flowerbeds instead of his designated spot. Then he began leaving muddy paw prints exclusively on the freshly mopped kitchen floor. His owners, bless their cotton socks, only responded with extra ear scratches and declarations of "Who’s a good boy?" which fueled his rebellious fire. He needed something truly outrageous.

One afternoon, staring longingly at the forbidden territory of the living room sofa, Bartholomew hatched a plan; he waited until the family left for groceries, then he jumped onto the sofa, opened his mouth wide, and swallowed the remote control whole, then he looked directly at the security camera and winked.

